## Generating Sub-panels for Figure-1

The following codes generate the subfigures that goes ti Fig. 1 in our manuscript.

* 1a. is manually created, showing an overview of the PARK framework
* 1b. demographic distributions are created using this code. We manually edited the legends of Fig. 1b4 to create a better visual.
* 1d. again, we created the graph using the code here, and manually edited the labels for better visual.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.patches import Patch
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.colors import LinearSegmentedColormap

In [ ]:
df = pd.read_csv('../../data/demographic_table.csv')

In [ ]:
df

In [ ]:
# The correct label for race is 'Race, n (%)'
# Now extract the corresponding rows
race_start_idx = df[df['Demographic Property'] == 'Race, n (%)'].index[0]
race_rows = df.iloc[race_start_idx : race_start_idx + 6].copy()

# Extract relevant columns and parse count and percentage
race_rows = race_rows[['Attribute', 'Total']]
race_rows[['Count', 'Percent']] = race_rows['Total'].str.extract(r'(\d+)\s*\(([\d.]+)%\)')
race_rows['Count'] = race_rows['Count'].astype(int)
race_rows['Percent'] = race_rows['Percent'].astype(float)

race_rows.reset_index(drop=True, inplace=True)
# remove row index 3 
# race_rows = race_rows.drop(index=3)
race_rows


In [ ]:
# Move "Not Mentioned" to the main pie instead of including in Non-White breakdown
# Split categories accordingly

white = race_rows[race_rows['Attribute'] == 'White'].copy()
not_mentioned = race_rows[race_rows['Attribute'] == 'Not Mentioned'].copy()
non_white = race_rows[
    ~race_rows['Attribute'].isin(['White', 'Not Mentioned'])
].copy()

# Prepare main pie chart data
main_labels = ['White', 'Non-White', 'Not Mentioned']
main_counts = [
    white['Count'].values[0],
    non_white['Count'].sum(),
    not_mentioned['Count'].values[0]
]
main_colors = ['#cfd8dc', '#90a4ae', '#e0e0e0']
main_explode = [0, 0.1, 0]

main_labels = ['White', 'Non-White', 'Unavailable']

# Prepare breakdown data
sub_labels = [f"{row['Attribute']} (n={row['Count']})" for _, row in non_white.iterrows()]
sub_counts = non_white['Count'].tolist()
sub_colors = ['#ffab91', '#4dd0e1', '#ffe082', '#ce93d8', '#a5d6a7']


# === Draw the final chart ===
fig, ax = plt.subplots(figsize=(9, 5))
ax.set_aspect('equal')

# # Pie chart with radius 0.75
# wedges, texts, autotexts = ax.pie(
#     main_counts,
#     labels=main_labels,
#     autopct=make_autopct(main_counts),
#     startangle=90,
#     radius=0.75,
#     colors=main_colors,
#     explode=main_explode,
#     wedgeprops=dict(edgecolor='white'),
#     textprops={'fontsize': 11}
# )

# # Adjust position of Non-White label again
# texts[1].set_position((0.68, 0.55))

# Pie chart with labels inside segments in format: Line 1 = label, Line 2 = count (percent)
def format_labels(labels, counts):
    total = sum(counts)
    return [
        f"{label}\n{count} ({count/total:.1%})"
        for label, count in zip(labels, counts)
    ]

formatted_main_labels = format_labels(main_labels, main_counts)

wedges, texts = ax.pie(
    main_counts,
    labels=formatted_main_labels,
    startangle=90,
    radius=0.85,
    colors=main_colors,
    explode=main_explode,
    labeldistance=0.6,
    wedgeprops=dict(edgecolor='white'),
    textprops={'fontsize': 16, 'ha': 'center', 'va': 'center', 'multialignment': 'center'}
)

texts[0].set_position((-0.45, -0.12))
texts[1].set_position((0.55, 0.37))
texts[2].set_position((0.2, 0.65))


# Bar chart shifted upward
x_bar = 1.25
bar_width = 0.25
bar_top = 0.8  # pull bar upward
sub_total = sum(sub_counts)
sub_fracs = [x / sub_total for x in sub_counts]
y_vals = np.cumsum([0] + sub_fracs)

for i in range(len(sub_labels)):
    y_bottom = bar_top - y_vals[i+1]
    height = sub_fracs[i]
    plt.bar(x_bar, height, width=bar_width, bottom=y_bottom, color=sub_colors[i], edgecolor='white')

# Connector lines updated
theta = (wedges[1].theta1 + wedges[1].theta2) / 2
x_pie = np.cos(np.radians(theta)) * 0.75
y_pie = np.sin(np.radians(theta)) * 0.75
bar_left_x = x_bar - bar_width / 2

for target_y in [bar_top - y_vals[-1], bar_top]:
    plt.plot([x_pie + 0.22, bar_left_x], [y_pie + 0.0, target_y], 'k-', lw=1.2)

# Legend underneath the bar
legend_elements = [Patch(facecolor=sub_colors[i], edgecolor='white', label=sub_labels[i]) for i in range(len(sub_labels))]
plt.legend(handles=legend_elements, loc='upper center', bbox_to_anchor=(0.7, 0.4),
           title="Non-White Subgroups", fontsize=15, title_fontsize=16, ncol=1)

# Final styling and save
plt.axis('equal')
plt.axis('off')
plt.tight_layout()
output_path = "plots/figure_1/race_distribution_pie.png"
plt.savefig(output_path, dpi=600, bbox_inches='tight')
plt.show()


In [ ]:
# Reload demographic table
df_demo = df.copy()

# Extract overall totals for PD and Non-PD from row 0
overall_row = df_demo.iloc[0]
pd_total_label = f"PD\n{overall_row['With PD']}"
non_pd_total_label = f"Non-PD\n{overall_row['Without PD']}"

# Extract sex breakdown from rows 1 and 2
sex_rows = df_demo.iloc[1:3].copy()
sex_rows = sex_rows[['Attribute', 'With PD', 'Without PD']]

print(sex_rows)

# Outer pie labels directly from the table values
outer_labels = [
    f"Female\n{sex_rows.iloc[0]['With PD']}",
    f"Male\n{sex_rows.iloc[1]['With PD']}",
    f"Female\n{sex_rows.iloc[0]['Without PD']}",
    f"Male\n{sex_rows.iloc[1]['Without PD']}"
]

# Parse counts for plotting
for col in ['With PD', 'Without PD']:
    sex_rows[col] = sex_rows[col].str.extract(r'(\d+)').astype(float)

with_pd_vals = sex_rows['With PD'].values
without_pd_vals = sex_rows['Without PD'].values
outer_counts = np.concatenate([with_pd_vals, without_pd_vals])
inner_counts = [with_pd_vals.sum(), without_pd_vals.sum()]

# Inner pie labels (from row 0)
inner_labels = [pd_total_label, non_pd_total_label]

# Define colors
inner_colors = ['#ef9a9a', '#a5d6a7']
outer_colors = ['#e57373', '#f48fb1', '#81c784', '#aed581']

# Draw nested pie chart
fig, ax = plt.subplots(figsize=(7, 7))

# Inner pie (PD vs Non-PD)
wedges, texts = ax.pie(
    inner_counts,
    labels=inner_labels,
    radius=0.5,
    colors=inner_colors,
    wedgeprops=dict(width=0.5, edgecolor='white'),
    labeldistance=0.5,
    textprops={'fontsize': 19, 'weight': 'bold', 'horizontalalignment': 'center', 'multialignment': 'center'}
)

texts[0].set_position((0.1, 0.2))
texts[1].set_position((0, -0.22))

# Outer pie (Male/Female breakdown)
wedges, texts = ax.pie(
    outer_counts,
    labels=outer_labels,
    radius=1.0,
    colors=outer_colors,
    labeldistance=0.75,
    wedgeprops=dict(width=0.5, edgecolor='white'),
    textprops={'fontsize': 20, 'horizontalalignment': 'center', 'multialignment': 'center'}
)

texts[0].set_position((0.62, 0.4))
# texts[0].set_rotation(-60)

# texts[2].set_position((-0.62, -0.4))
# texts[2].set_rotation(-60)

plt.tight_layout()

# Save the figure
output_path = "plots/figure_1/sex_distribution_nested_pie.png"
plt.savefig(output_path, dpi=600, bbox_inches='tight')
plt.show()


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.colors import LinearSegmentedColormap

# Load the data
df_stage = pd.read_csv("../../data/df_stage_data.csv")

# Define total PD count (manually or from external source)
total_pd_count = 670  # as per dataset

# Extract numeric score
df_stage['Numeric Score'] = df_stage['Stage Label'].str.extract(r'(\d+\.?\d*)').astype(float)

# Count stage occurrences
stage_counts = df_stage.groupby(['Stage Label', 'Numeric Score']).size().reset_index(name='count')
stage_counts = stage_counts.sort_values('Numeric Score')

# Group Stage 4 and 5 together
collapsed_stage_counts = stage_counts.copy()
collapsed_stage_counts['Stage Label'] = collapsed_stage_counts['Stage Label'].replace(
    ['Stage 4', 'Stage 5'], 'Stage 4 and 5'
)
collapsed_stage_counts['Numeric Score'] = collapsed_stage_counts.apply(
    lambda row: 4.5 if row['Stage Label'] == 'Stage 4 and 5' else row['Numeric Score'], axis=1
)

# Regroup and sort
grouped = collapsed_stage_counts.groupby(['Stage Label', 'Numeric Score']).agg({'count': 'sum'}).reset_index()
grouped = grouped.sort_values('Numeric Score')

# Calculate number of PDs with unavailable stage info
known_total = grouped['count'].sum()
unknown_count = total_pd_count - known_total

# Add "Unavailable" as a wedge
if unknown_count > 0:
    grouped = pd.concat([grouped, pd.DataFrame([{
        'Stage Label': 'Unavailable',
        'Numeric Score': 5.5,
        'count': unknown_count
    }])], ignore_index=True)

# Create labels
stage_labels = [
    f"{row['Stage Label']}" for _, row in grouped.iterrows()
]

# Define colormap starting with #81c784
custom_cmap = LinearSegmentedColormap.from_list("green_to_red", [
    "#81c784", "#fdbf6f", "#e53935"
])

# Generate colors and add gray for unavailable
min_score = grouped['Numeric Score'].min()
max_score = grouped['Numeric Score'].max()
colors = [
    '#bdbdbd' if row['Stage Label'] == 'Unavailable'
    else custom_cmap((row['Numeric Score'] - min_score) / (max_score - min_score))
    for _, row in grouped.iterrows()
]

# Explode stage 4 and 5 and unavailable
explode = [
    0.08 if row['Stage Label'] in ['Stage 4 and 5', 'Unavailable'] else 0
    for _, row in grouped.iterrows()
]

# Draw pie chart
fig, ax = plt.subplots(figsize=(7, 7))
wedges, texts = ax.pie(
    grouped['count'],
    labels=None,
    colors=colors,
    explode=explode,
    startangle=30,
    counterclock=False,
    wedgeprops=dict(edgecolor='white'),
)

# Add labels (inside for 0–3, outside for exploded or unavailable)
for wedge, row in zip(wedges, grouped.itertuples()):
    ang = (wedge.theta2 + wedge.theta1) / 2
    x, y = np.cos(np.deg2rad(ang)), np.sin(np.deg2rad(ang))
    label = f""

    if row._1 in ['Stage 4 and 5']:
        ax.text(1.15 * x, 1.2 * y, label, ha='center', va='center',
                fontsize=20, multialignment='center')
    else:
        ax.text(0.75 * x, 0.7 * y, label, ha='center', va='center',
                fontsize=20, multialignment='center', color='black')

# Title and layout
plt.tight_layout()

# Save
output_path = "plots/figure_1/stage_distribution_pie_no_label.png"
plt.savefig(output_path, dpi=600, bbox_inches='tight')
plt.show()


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# === Load and process data ===
df_demo = df.copy()
age_start_idx = df_demo[df_demo['Demographic Property'].str.contains("Age in years", na=False)].index[0]
age_rows = df_demo.iloc[age_start_idx : age_start_idx + 6].copy()
age_rows = age_rows[['Attribute', 'With PD', 'Without PD']]

# Parse counts
for col in ['With PD', 'Without PD']:
    age_rows[col] = age_rows[col].str.extract(r'(\d+)').astype(float)

age_labels = age_rows['Attribute'].str.replace('years', '').str.strip().tolist()
with_pd_vals = age_rows['With PD'].values
without_pd_vals = age_rows['Without PD'].values

# === Plot ===
width = 0.45
x = np.linspace(0, len(age_labels) - 1, len(age_labels)) * 0.6  # tighter spacing

fig, ax = plt.subplots(figsize=(5, 4))
bars_non_pd = ax.bar(x, without_pd_vals, width, label='Non-PD', color='#a5d6a7', edgecolor='black', linewidth=0.3)
bars_pd = ax.bar(x, with_pd_vals, width, bottom=without_pd_vals, label='PD', color='#ef9a9a', edgecolor='black', linewidth=0.3) 

# Tick and label settings
ax.set_xticks(x)
ax.set_xticklabels(age_labels, fontsize=18, rotation=25)
ax.set_xlim(min(x) - 0.3, max(x) + 0.3)


# Axis settings
ax.set_xticks(x)
ax.set_xticklabels(age_labels, fontsize=13)
ax.set_ylabel("Participants", fontsize=14)
ax.legend(title="Group", fontsize=12, title_fontsize=12, loc="upper right")

# Optional value labels on top
for rects in bars_pd:
    height = rects.get_height() + rects.get_y()
    ax.text(rects.get_x() + rects.get_width() / 2, height + 5, f'{int(height)}',
            ha='center', va='bottom', fontsize=14)

# Grid and styling
ax.grid(axis='y', linestyle='--', alpha=0.4)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig("plots/figure_1/age_distribution_pd_vs_nonpd_bar_chart.png", dpi=300, bbox_inches='tight')
plt.show()


In [ ]:
import plotly.graph_objects as go

labels = [
    "Parktest_Unusable", # 0
    "Parktest", # 1
    "Super PD", # 2
    "Rooster PD_Unusable", # 3
    "Rooster PD", # 4
    "Cluster PD", # 5
    "PD Care Facility_Unusable", # 6
    "PD Care Facility", # 7
    "Route PD", # 8
    "Validation (s)", # 9
    "Validation (u)", # 10
    "Model Building", # 11
    "Training + Development", # 12
    "Balanced Test", # 13
    "External Evaluation" # 14
]

labels = [
    "",
    "",
    "",
    "",
    "",
    "",
    "",
    "",
    "",
    "",
    "",
    "",
    "",
    "",
    "",
    "",
]

# Define flow structure
source = [
    1, 2, 4, 5, 7, 8, # Study 1 to 6 will go Model Building 
    11, 11,              # Model Building will go to Train + Dev and also Balanced Test
    9, 10               # Study 7 and 8 will go to External Evaluation
]
target = [
    11, 11, 11, 11, 11, 11, # Study 1 to 6 will go Model Building 
    12, 13,            # Model Building will go to Train + Dev and also Balanced Test
    14, 14            # Study 7 and 8 will go to External Evaluation
]

# Study_Name
# 1. Parktest                     1102
# 1. Parktest_Unusable              75
# 2. Super PD                      103
# 3. Rooster PD                    167
# 3. Rooster PD_Unusable             1
# 4. Cluster PD                     83
# 5. PD Care Facility              173
# 5. PD Care Facility_Unusable       6
# 6. Route PD                       48
# 7. Validation (s)                100
# 8. Validation (u)                 89

# Visual thickness of each path
value = [
    1102, 103, 167, 83, 173, 48,
    1400, 200,
    100, 89
]

supervised = '#C4EC99'
unsupervised = '#FFB37B'
on_demand_supervision = '#47DBF5'

# Assign node colors (extend if needed)
node_colors = [
    '#FFB37B', # 0
    '#FFB37B', # 1
    '#C4EC99', # 2
    '#C4EC99', # 3
    '#C4EC99', # 4
    '#C4EC99', # 5
    '#47DBF5', # 6
   '#47DBF5', # 7
    '#47DBF5', # 8
    '#C4EC99', # 9
    '#FFB37B', # 10
    "#1F77B4", # 11
    "#33D6AB", # 12
   "#830202", # 13
    "#9467BD" # 14
]

fig = go.Figure(data=[go.Sankey(
    arrangement="snap",
    node=dict(
        pad=15,
        thickness=80,
        line=dict(color="black", width=0.5),
        label=labels,
        color=node_colors
    ),
    link=dict(
        source=source,
        target=target,
        value=value,
        color="rgba(255, 255, 255)"
    )
)])

fig.update_layout(
    font_size=20,
    height=1100,
    width=900,  # reduce this number for tighter layout
    margin=dict(l=10, r=10, t=40, b=10)
)

# Save high-resolution image if needed
fig.write_image("plots/figure_1/participant_flow_sketch.png", scale=3)

fig.show()
